# Zero-shot XGBoost — Binary Classification Benchmark

Exhaustive comparison across five ADMET binary classification datasets from the
[Therapeutics Data Commons](https://tdcommons.ai/).

| Dataset | Task | n | Imbalance |
|---|---|---|---|
| BBB_Martins | Blood-brain barrier penetration | 2,050 | 3.3 : 1 |
| Pgp_Broccatelli | P-glycoprotein inhibition | 1,218 | 1.1 : 1 |
| hERG | hERG cardiotoxicity | 655 | 2.2 : 1 |
| HIA_Hou | Human intestinal absorption | 578 | 6.4 : 1 |
| AMES | Ames mutagenicity | 7,278 | 1.2 : 1 |

**Models compared:**
- `ZeroShotXGBClassifier` — no tuning, parameters derived from dataset statistics
- Default XGBoost (`XGBClassifier` with scale_pos_weight adjusted for imbalance)
- Default Random Forest (`RandomForestClassifier` with class_weight='balanced')
- Logistic Regression (L2, class_weight='balanced')

**Features:** 1 024-bit ECFP4 Morgan fingerprints computed with RDKit.

**Evaluation:** Stratified 70 / 20 test split, ROC-AUC and PR-AUC on the held-out test set.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from zsxgboost import ZeroShotXGBClassifier

_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def smiles_to_ecfp4(smiles_list, n_bits=1024):
    """Convert SMILES strings to binary ECFP4 fingerprint matrix."""
    gen = _morgan_gen if n_bits == 1024 else rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            fps.append(np.zeros(n_bits, dtype=np.uint8))
        else:
            fps.append(gen.GetFingerprintAsNumPy(mol).astype(np.uint8))
    return np.vstack(fps)


def load_dataset(path, test_size=0.2, random_state=42):
    """Load a .tab file, compute fingerprints, and split into train/test."""
    df = pd.read_csv(path, sep='\t')
    df = df.dropna(subset=['Drug', 'Y'])
    df['Y'] = df['Y'].astype(int)
    train_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df['Y'], random_state=random_state
    )
    X_train = smiles_to_ecfp4(train_df['Drug'])
    X_test  = smiles_to_ecfp4(test_df['Drug'])
    y_train = train_df['Y'].values
    y_test  = test_df['Y'].values
    return X_train, X_test, y_train, y_test


print("Imports OK")

In [ ]:
# Uncomment to install if needed
# !pip install rdkit zsxgboost xgboost scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from zsxgboost import ZeroShotXGBClassifier


def smiles_to_ecfp4(smiles_list, n_bits=1024):
    """Convert SMILES strings to binary ECFP4 fingerprint matrix."""
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            fps.append(np.zeros(n_bits, dtype=np.uint8))
        else:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
            fps.append(np.frombuffer(fp.ToBitString().encode(), dtype='u1') - ord('0'))
    return np.vstack(fps).astype(np.uint8)


def load_dataset(path, test_size=0.2, random_state=42):
    """Load a .tab file, compute fingerprints, and split into train/test."""
    df = pd.read_csv(path, sep='\t')
    df = df.dropna(subset=['Drug', 'Y'])
    df['Y'] = df['Y'].astype(int)
    train_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df['Y'], random_state=random_state
    )
    X_train = smiles_to_ecfp4(train_df['Drug'])
    X_test  = smiles_to_ecfp4(test_df['Drug'])
    y_train = train_df['Y'].values
    y_test  = test_df['Y'].values
    return X_train, X_test, y_train, y_test


print("Imports OK")

## 2. Dataset configuration

In [ ]:
DATASETS = [
    {"name": "BBB_Martins",      "file": "../data/bbb_martins.tab",      "pos_label": "BBB+",    "neg_label": "BBB−"},
    {"name": "Pgp_Broccatelli",  "file": "../data/pgp_broccatelli.tab",  "pos_label": "inhibitor","neg_label": "non-inh"},
    {"name": "hERG",             "file": "../data/herg.tab",             "pos_label": "blocker",  "neg_label": "non-block"},
    {"name": "HIA_Hou",          "file": "../data/hia_hou.tab",          "pos_label": "absorbed", "neg_label": "non-abs"},
    {"name": "AMES",             "file": "../data/ames.tab",             "pos_label": "mutagen",  "neg_label": "non-mut"},
]

print(f"Datasets: {[d['name'] for d in DATASETS]}")

## 3. Define models

In [ ]:
def make_models(imbalance_ratio):
    """Return a dict of model-name → unfitted estimator."""
    return {
        "Zero-shot XGBoost": ZeroShotXGBClassifier(verbose=False),
        "Default XGBoost":   XGBClassifier(
            scale_pos_weight=imbalance_ratio,   # only imbalance correction applied
            eval_metric="auc",
            verbosity=0,
            random_state=42,
        ),
        "Default RF":        RandomForestClassifier(class_weight="balanced", random_state=42),
        "Logistic Reg.":     LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    }

## 4. Run benchmark

In [ ]:
# Store results and curve data for later plotting
all_results   = []   # [{dataset, model, roc_auc, pr_auc}, ...]
all_curves    = {}   # {dataset: {model: {fpr, tpr, prec, rec}}}
all_profiles  = {}   # {dataset: DatasetProfile}

MODEL_STYLES = {
    "Zero-shot XGBoost": dict(lw=2.5, linestyle="-",   color="#e41a1c"),
    "Default XGBoost":   dict(lw=1.5, linestyle="-.",  color="#377eb8"),
    "Default RF":        dict(lw=1.5, linestyle="--",  color="#4daf4a"),
    "Logistic Reg.":     dict(lw=1.5, linestyle=":",   color="#984ea3"),
}

for ds in DATASETS:
    name = ds["name"]
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test = load_dataset(ds["file"])

    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    imbalance_ratio = neg / pos
    baseline_pr = y_test.mean()

    print(f"  Train n={len(y_train):,}  Test n={len(y_test):,}")
    print(f"  Imbalance (neg/pos): {imbalance_ratio:.2f}")

    all_curves[name] = {}

    models = make_models(imbalance_ratio)

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = model.decision_function(X_test)

        roc_auc = roc_auc_score(y_test, y_prob)
        pr_auc  = average_precision_score(y_test, y_prob)

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        prec, rec, _ = precision_recall_curve(y_test, y_prob)

        all_results.append({"Dataset": name, "Model": model_name,
                             "ROC-AUC": roc_auc, "PR-AUC": pr_auc})
        all_curves[name][model_name] = dict(fpr=fpr, tpr=tpr, prec=prec, rec=rec,
                                            baseline_pr=baseline_pr)

        if model_name == "Zero-shot XGBoost":
            all_profiles[name] = model.profile_

        print(f"  {model_name:22s}  ROC-AUC={roc_auc:.4f}  PR-AUC={pr_auc:.4f}")

results_df = pd.DataFrame(all_results)
print("\nBenchmark complete.")

## 5. Dataset profiles (zero-shot inspection)

In [ ]:
for ds_name, profile in all_profiles.items():
    print(f"\n--- {ds_name} ---")
    print(profile)

## 6. Summary table

In [ ]:
pivot_roc = results_df.pivot(index='Dataset', columns='Model', values='ROC-AUC')
pivot_pr  = results_df.pivot(index='Dataset', columns='Model', values='PR-AUC')

# Order datasets as defined, order columns consistently
col_order = ["Zero-shot XGBoost", "Default XGBoost", "Default RF", "Logistic Reg."]
ds_order  = [d['name'] for d in DATASETS]
pivot_roc = pivot_roc.loc[ds_order, col_order]
pivot_pr  = pivot_pr.loc[ds_order, col_order]

print("=== ROC-AUC ===")
print(pivot_roc.to_string(float_format="{:.4f}".format))
print()
print("=== PR-AUC ===")
print(pivot_pr.to_string(float_format="{:.4f}".format))

## 7. Summary heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))

for ax, pivot, title in zip(axes, [pivot_roc, pivot_pr], ["ROC-AUC", "PR-AUC"]):
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                   vmin=pivot.values.min() - 0.02,
                   vmax=pivot.values.max() + 0.02)
    ax.set_xticks(range(len(col_order)))
    ax.set_xticklabels(col_order, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(ds_order)))
    ax.set_yticklabels(ds_order, fontsize=9)
    ax.set_title(title, fontweight='bold')
    for i in range(len(ds_order)):
        for j in range(len(col_order)):
            ax.text(j, i, f"{pivot.values[i, j]:.3f}",
                    ha='center', va='center', fontsize=8.5,
                    color='black')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

plt.suptitle("Model comparison across ADMET binary classification datasets (ECFP4)",
             fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig("../results/benchmark_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. ROC and PR curves per dataset

In [ ]:
n_ds = len(DATASETS)
fig, axes = plt.subplots(n_ds, 2, figsize=(12, 4 * n_ds))

for row, ds in enumerate(DATASETS):
    name   = ds['name']
    curves = all_curves[name]

    ax_roc = axes[row, 0]
    ax_pr  = axes[row, 1]

    for model_name, style in MODEL_STYLES.items():
        c = curves[model_name]
        roc_auc = roc_auc_score(
            [0] * len(c['fpr']),   # placeholder; extract from results_df
            [0] * len(c['fpr'])
        )
        # Get stored metrics
        row_df = results_df[(results_df['Dataset'] == name) & (results_df['Model'] == model_name)].iloc[0]
        auc_label = f"{model_name} (AUC={row_df['ROC-AUC']:.3f})"
        ap_label  = f"{model_name} (AP={row_df['PR-AUC']:.3f})"

        ax_roc.plot(c['fpr'], c['tpr'], label=auc_label, **style)
        ax_pr.plot(c['rec'], c['prec'], label=ap_label,  **style)

    ax_roc.plot([0, 1], [0, 1], color='grey', lw=0.8, linestyle=':')
    ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
    ax_roc.set_title(f'{name} — ROC', fontweight='bold')
    ax_roc.legend(fontsize=7.5)

    bp = curves[list(curves.keys())[0]]['baseline_pr']
    ax_pr.axhline(bp, color='grey', lw=0.8, linestyle=':', label=f'Baseline ({bp:.2f})')
    ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
    ax_pr.set_title(f'{name} — Precision-Recall', fontweight='bold')
    ax_pr.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig("../results/benchmark_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Mean rank across datasets

In [ ]:
# Rank models within each dataset (1 = best)
for metric in ['ROC-AUC', 'PR-AUC']:
    pivot = results_df.pivot(index='Dataset', columns='Model', values=metric).loc[ds_order, col_order]
    ranks = pivot.rank(axis=1, ascending=False)
    mean_ranks = ranks.mean().sort_values()
    print(f"\nMean rank by {metric} (lower = better):")
    for model, rank in mean_ranks.items():
        print(f"  {model:22s}: {rank:.2f}")

## 10. Bar chart comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = [MODEL_STYLES[m]['color'] for m in col_order]
x = np.arange(len(ds_order))
width = 0.2
offsets = np.linspace(-1.5, 1.5, len(col_order)) * width

for ax, metric in zip(axes, ['ROC-AUC', 'PR-AUC']):
    pivot = results_df.pivot(index='Dataset', columns='Model', values=metric).loc[ds_order, col_order]
    for j, (model, color, offset) in enumerate(zip(col_order, colors, offsets)):
        ax.bar(x + offset, pivot[model], width, label=model, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(ds_order, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0.5, 1.02)
    ax.legend(fontsize=8)
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)

plt.suptitle("Model comparison — ADMET binary classification (ECFP4)", fontweight='bold')
plt.tight_layout()
plt.savefig("../results/benchmark_bars.png", dpi=150, bbox_inches='tight')
plt.show()